In [1]:
from optimization import *

device = torch.device('cuda:0')

In [2]:
config = OptimizationConfig(
    n_batch=1,
    min_steps=1000,
    max_steps=2000,
    sa_steps=2000,
    seq_length=100,
    alt_start_pos=50,
    w_mrl=1.0,
    w_prot=1.0,
    w_edit=1.0,
    max_fix_aa_grad_norm_factor=1.0,
    max_fix_edit_grad_norm_factor=0.1,
)

#Random AA for testing
fix_aa = F.one_hot(torch.randint(20, (1, 50//3 - 1)), 21).transpose(2, 1).to(torch.float32).to(config.device).repeat(config.n_batch, 1, 1) 
right_overhang_mask = F.one_hot(torch.randint(4, (1, 2)), 4).transpose(2, 1).to(torch.bool).to(config.device).repeat(config.n_batch, 1, 1)
seed_onehot = F.pad(F.one_hot(torch.randint(4, (1, 50)), 4).transpose(2, 1), (0, 50)).to(torch.float32).to(config.device).repeat(config.n_batch, 1, 1)

print(''.join([ AA_ALPHABET[i] for i in fix_aa[0].argmax(0) ]))
print(''.join([ DNA_ALPHABET[i] for i in right_overhang_mask[0].float().argmax(0) ]))

IGNFVPVEWRYESWW
TC


In [4]:
# Create and run pipeline
pipeline = OptimusOLGPipeline(config)
result = pipeline.run(fix_aa, right_overhang_mask, seed_onehot)

Starting gradient-based optimization...


/usr/local/lib/python3.12/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  return F.conv1d(


Starting simulated annealing with 1 sequences...
Optimization complete. Found 1 optimized sequences.


In [ ]:
pipeline.plot_results(result)

In [6]:
torch.stack([ torch.stack([r[1], r[2]]) for r in result.best_results ])

tensor([[1.2309, 1.0995]], device='cuda:0')

In [7]:
[ ''.join([ DNA_ALPHABET[c] for c in r[0].argmax(0) ]) for r in result.best_results ][0]

'GTCAAATAACAGTTAATAGGGGAGTATATTTCCCTGTGTGTCATTCTCTAATGATAGGCAATTTCGTTCCTGTCGAGTGGAGATACGAATCATGGTGGTC'

In [8]:
''.join([ DNA_ALPHABET[c] for c in seed_onehot[0].argmax(0) ])

'GTCGTCTAACAGTTAATACGGGCTTGTATCTACATGTGTATCACCCATTCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA'